# Classifiers evaluation

In [ ]:
#%pip install numpy pandas matplotlib scikit-learn --upgrade

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams['figure.dpi'] = 120.0  #size of figures

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## 1. Imbalanced data and cost-based performance

**Data loading & pre-processing**

[Statlog German Credit](https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data) dataset: 24 features, target `1` = good client, `2` = bad client. See `german.doc` for the description.

Note: Some of the features are categorical, but we treat them as numeric. For simplicity and with disregard for the model's performance, they are not handled properly (more on that topic on Class 3).

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data-numeric"
german = pd.read_csv(url, sep=r"\s+", header=None)
german = german.rename(columns={24: "target"})
german["target"] = german["target"] - 1  # recode target: 0 = good, 1 = bad
german

In [ ]:
german["target"].value_counts(normalize=True)

Balance of the target's classes matters. If the minority class is severely underrepresented it may affect model's stability and performance.

There are few approaches to tackle the issue:
- undersampling the majority class,
- oversampling the minority class – by duplicating records or by generating synthetic ones (e.g. SMOTE),
- algorithmic (cost-sensitive) learning, e.g. `class_weight="balanced"` in many scikit-learn models,
- **cost-based choice of the cutoff** (used below) – strictly speaking it addresses *unequal misclassification costs* rather than imbalance itself, but the two usually go together.

Check [imbalanced-learn](https://imbalanced-learn.org/stable/user_guide.html#user-guide) documentation for details.

In [ ]:
X = german.drop(columns="target")
y = german["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
print(X_train.shape, X_val.shape,)
print(y_train.mean(), y_val.mean())  # share of bad clients in each subset

**Building logistic regression model**

[Logistic regression in scikit-learn](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)

In [ ]:
model_lr = LogisticRegression(penalty=None, max_iter=1000)
model_lr.fit(X_train, y_train)

In [ ]:
pd.Series(model_lr.coef_[0], index=X.columns), model_lr.intercept_

**Predict probability of bad credit**

In [ ]:
score_train = model_lr.predict_proba(X_train)[:, 1]
score_val = model_lr.predict_proba(X_val)[:, 1]

**Cost-based approach in model assessment**

The cost matrix has the same layout as scikit-learn's `confusion_matrix`: rows = actual class, columns = predicted class.

|              | predicted good (0) | predicted bad (1) |
|--------------|:---:|:---:|
| **actual good (0)** | 0 | 1 |
| **actual bad (1)**  | 5 | 0 |

Granting credit to a bad client costs 5 times more than rejecting a good one.

If predicted probabilities are well calibrated, the cost-minimising cutoff is

$$p^* = \frac{C_{FP}}{C_{FP} + C_{FN}} = \frac{1}{1 + 5} \approx 0.167$$

i.e. we should flag a client as bad already when the probability of being bad exceeds ~17%.

In [ ]:
def cutoff_analysis(
    y_true: pd.Series,
    y_score: np.ndarray,
    cost_matrix: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    For cutoffs between 0 and 1 (step 0.01) convert predicted probabilities `y_score`
    into labels (1 if score >= cutoff) and compare them with true labels `y_true`.

    Returns the cutoffs and the accuracy for each cutoff. If `cost_matrix` is given
    (rows = actual class, columns = predicted class), returns the average cost
    per observation instead of accuracy.
    """
    cutoffs = np.arange(0, 1.0, 0.01)
    values = []
    for cutoff in cutoffs:
        y_pred = (y_score >= cutoff).astype(int)
        conf_mat = confusion_matrix(y_true, y_pred, labels=[0, 1])  # always 2x2
        if cost_matrix is None:
            values.append(np.trace(conf_mat) / conf_mat.sum())
        else:
            values.append((conf_mat * cost_matrix).sum() / conf_mat.sum())
    return cutoffs, np.array(values)

In [ ]:
cost_matrix = np.array([[0, 1],
                        [5, 0]])
cutoffs, cost_train = cutoff_analysis(y_train, score_train, cost_matrix)
_, cost_val = cutoff_analysis(y_val, score_val, cost_matrix)
cost_matrix

In [ ]:
best_train = np.argmin(cost_train)
best_val = np.argmin(cost_val)
theoretical_cutoff = cost_matrix[0, 1] / (cost_matrix[0, 1] + cost_matrix[1, 0])

plt.figure()
plt.xlabel("Cutoff")
plt.ylabel("Cost per client")
plt.title("Cost vs. cutoff")

plt.plot(cutoffs, cost_train, color="red", label="Train")
plt.plot(cutoffs, cost_val, color="tab:blue", label="Validation")
plt.scatter(cutoffs[best_train], cost_train[best_train], color="red",
            label=f"Train min = {cost_train[best_train]:.3f} at cutoff = {cutoffs[best_train]:.2f}")
plt.scatter(cutoffs[best_val], cost_val[best_val], color="tab:blue",
            label=f"Val min = {cost_val[best_val]:.3f} at cutoff = {cutoffs[best_val]:.2f}")
plt.axvline(theoretical_cutoff, color="green", linestyle="--",
            label=f"Theoretical cutoff = {theoretical_cutoff:.3f}")

plt.legend();

Observations:
- The cost on the training set is lower than on the validation set – the model fits the data it was trained on slightly better, i.e. it may **overfit** slightly.
- The validation curve is noisy (only 200 clients), so its minimum is not a very precise estimate of the best cutoff. In practice, cross-validation gives a more stable choice.
- Both minima lie far below the default cutoff of 0.5 – with asymmetric costs the default cutoff is rarely optimal.

## 2. Assessing model performance

[Statlog Australian Credit Approval](https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval) dataset: 14 anonymised features, `class` = 1 if credit was approved.

Pre-processing:
- `V3`, `V4`, `V5` and `V11` are **categorical** (coded as integers 1, 2, 3, ...). Treating such codes as numbers would impose an artificial order, but we disregard the issue for now.
- `V13` is strongly right-skewed (values from 1 to 100 001), so we take the logarithm.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/australian/australian.dat"
dataset = pd.read_csv(url, sep=" ", header=None, names=[f"V{i}" for i in range(15)])
dataset = dataset.rename(columns={"V14": "class"})

dataset["V13"] = np.log(dataset["V13"])
dataset

**Data split using sklearn**

In [ ]:
X = dataset.drop(columns="class")
y = dataset["class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

**Confusion matrix**

❗ Remember class indicator (0, 1,...) and actual or predicted values may be switched in confusion matrix - always check the axes.

<img src="https://miro.medium.com/max/712/1*Z54JgbS4DUwWSknhDCvNTQ.png" width=400>

<img src="https://miro.medium.com/max/1780/1*LQ1YMKBlbDhH9K6Ujz8QTw.jpeg"  width=400>

**Performance measures derived from confusion matrix:**

- Accuracy - percentage of correct predictions

$$ACC = \frac{TP + TN}{TP + FP + TN + FN}$$

- Precision - percentage of positive predictions which were actually correct

$$PREC = \frac{TP}{TP + FP}$$

- Recall - what percentage of actual positives were predicted correctly
 (Recall = Sensitivity = Hit rate = True Positive Rate (TPR))

$$REC = \frac{TP}{TP + FN}$$

- F1 Score - traditional F-measure or balanced F-score, the harmonic mean of precision and recall

$$F_1 = 2 \cdot \frac{PREC \cdot REC}{PREC + REC}$$

- Specificity (True Negative Rate) - what percentage of actual negatives were predicted correctly

$$SPEC = \frac{TN}{TN + FP}$$

- False Positive Rate - what percentage of actual negatives were wrongly predicted as positive (used in the ROC curve)

$$FPR = \frac{FP}{FP + TN} = 1 - SPEC$$

In [ ]:
model2 = LogisticRegression(penalty=None, max_iter=1000)
model2.fit(X_train, y_train)

In [ ]:
y_test_pred = model2.predict(X_test)

In [ ]:
confm = confusion_matrix(y_test, y_test_pred)
confm

In [ ]:
#What is what in confusion matrix?
confusion_matrix?

In [ ]:
tn, fp, fn, tp = confm.ravel()
ACC = (tp + tn) / (tp + fp + tn + fn)
PREC = tp / (tp + fp)
REC = tp / (tp + fn)
F1 = 2 * PREC * REC / (PREC + REC)
SPEC = tn / (tn + fp)
print(f"ACC  {ACC:.3f}\nPREC {PREC:.3f}\nREC  {REC:.3f}\nF1   {F1:.3f}\nSPEC {SPEC:.3f}")

In [ ]:
classification_report?

In [ ]:
# Sklearn built-in report
print(classification_report(y_test, y_test_pred))

❗ The test set has only 138 observations, so these metrics depend noticeably on which observations were drawn (try a different `random_state`). Cross-validation (e.g. `cross_val_score`) gives a more reliable estimate – for this model the 10-fold CV accuracy is about 0.85. We'll discuss cross-validation on Class 3.

### Visual analysis of a model - ROC Curve

**ROC curve + AUC**

In [ ]:
y_test_proba = model2.predict_proba(X_test)[:, 1]
y_train_proba = model2.predict_proba(X_train)[:, 1]

In [ ]:
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba)
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
auc_test = auc(fpr_test, tpr_test)
auc_train = auc(fpr_train, tpr_train)

plt.figure()

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC & AUC")

plt.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Random, AUC = 0.5")
plt.plot([0, 0, 1], [0, 1, 1], color="navy", linestyle=":", label="Perfect classifier, AUC = 1.0")

plt.plot(fpr_train, tpr_train, color="orange", label=f"Model - train, AUC = {auc_train:.2f}")
plt.plot(fpr_test, tpr_test, color="red", label=f"Model - test, AUC = {auc_test:.2f}")
plt.legend(loc="lower right");

## Exercises

Load Iris dataset from https://raw.githubusercontent.com/uiuc-cse/data-fa14/gh-pages/data/iris.csv to 'iris' DataFrame

Code `species` column to have value 1 if iris is from _versicolor_ species and 0 otherwise

Your goal is to predict `species` column. Split dataset to train and validation subsets using `train_test_split` function. Training set should have **75%** of all observations. Use `random_state=42` and stratify by `species`.

Build logistic regression (with `LogisticRegression` from `sklearn`) using **Elastic-net** regularization with 0.35 L1 ratio. Note: only one solver supports Elastic-net.

Make a prediction on the validation set with a 0.5 cutoff threshold. Produce classification report with `classification_report`. What is accuracy of the model?

Why is the accuracy so low? Check [Iris dimensionality example](https://jakevdp.github.io/PythonDataScienceHandbook/05.02-introducing-scikit-learn.html#Unsupervised-learning-example:-Iris-dimensionality) in the Python Data Science Handbook.